# EDA Monitoring Logs

Exploratory analysis for monitoring logs and metrics.

Steps:
- Inventory log directories and sizes.
- Preview recent log lines for key files.
- Run the training data audit and summarize outputs.



In [ ]:
from __future__ import annotations

import json
import os
import sys
import subprocess
from pathlib import Path

# Resolve repo root from the notebook location.
REPO_ROOT = Path.cwd()
for parent in [REPO_ROOT] + list(REPO_ROOT.parents):
    if (parent / 'scripts').exists() and (parent / 'notebooks').exists():
        REPO_ROOT = parent
        break

# Ensure local modules are importable.
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'src'))

PY = sys.executable

def run(cmd: list[str]) -> None:
    # Run a command from the repo root with PYTHONPATH set.
    env = os.environ.copy()
    env['PYTHONPATH'] = os.pathsep.join([str(REPO_ROOT / 'src'), str(REPO_ROOT)])
    print('$', ' '.join(cmd))
    subprocess.run(cmd, cwd=str(REPO_ROOT), check=True, env=env)

def show_json(rel_path: str) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        print(path.read_text(encoding='utf-8', errors='ignore')[:2000])
        return
    print(json.dumps(data, indent=2))

def list_dir(rel_path: str, limit: int = 20) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    print(f'\n{rel_path}/')
    for item in sorted(path.iterdir())[:limit]:
        print(' -', item.name)


In [ ]:
from pathlib import Path

log_root = REPO_ROOT / 'logs'
experiments_root = REPO_ROOT / 'experiments'

summary = {
    'logs': {},
    'experiments': {},
}

print('Logs root:', log_root)
if log_root.exists():
    log_files = [p for p in log_root.rglob('*') if p.is_file()]
    summary['logs']['file_count'] = len(log_files)
    print('Log files:', len(log_files))
    for sample in log_files[:10]:
        print(' -', sample.relative_to(REPO_ROOT))
else:
    print('Missing logs directory')

print('Experiments root:', experiments_root)
if experiments_root.exists():
    metric_files = [p for p in experiments_root.rglob('metrics.json') if p.is_file()]
    summary['experiments']['metrics_files'] = len(metric_files)
    for sample in metric_files[:8]:
        print(' -', sample.relative_to(REPO_ROOT))


In [ ]:
# Preview recent log lines for selected logs.
if log_root.exists():
    candidates = sorted([p for p in log_root.iterdir() if p.is_file()])
    for path in candidates[:3]:
        print('')
        print(f'Tail of {path.name}:')
        try:
            lines = path.read_text(encoding='utf-8', errors='ignore').splitlines()
        except Exception:
            lines = []
        for line in lines[-20:]:
            print(line)


In [ ]:
# Persist summary for quick reference.
report_dir = REPO_ROOT / 'reports'
report_dir.mkdir(parents=True, exist_ok=True)
summary_path = report_dir / 'eda_monitoring_logs_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))
print('Saved summary to', summary_path)


In [ ]:
# Generate a training data audit
run([PY, 'scripts/training_data_audit.py'])



In [ ]:
# Summarize monitor-related entries from the training data audit.
audit_path = REPO_ROOT / 'reports' / 'TRAINING_DATA.json'
if not audit_path.exists():
    print('Missing:', audit_path)
else:
    audit = json.loads(audit_path.read_text(encoding='utf-8'))
    items = [
        item for item in audit.get('required', []) + audit.get('optional', [])
        if 'monitor' in str(item.get('name', '')).lower()
    ]
    if not items:
        print('No monitor entries found in TRAINING_DATA.json')
    else:
        print('monitor datasets in audit:')
        for item in items:
            print(' -', item.get('name'), '|', item.get('status'), '|', item.get('path'))


In [ ]:
# Quick artifact index for verification.
for folder in ['models', 'experiments', 'artifacts', 'runs', 'reports', 'logs']:
    path = REPO_ROOT / folder
    if not path.exists():
        continue
    print(f'\n{folder}/')
    for item in sorted(path.iterdir())[:20]:
        print(' -', item.name)
